In [1]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_gizmosql import *

In [4]:
p(f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18'])).group_by(c('place_type'), c('place_of_publication')).agg(nw.len()).sort('len', descending=True)).to_pandas()

,place_type,place_of_publication,len
0,standardized,Leipzig,121300
1,unstandardized,[S.l.],92437
2,unstandardized,Leipzig,75037
3,standardized,Wittenberg,51976
4,standardized,Jena,46039
...,...,...,...
19956,unstandardized,Zu Lubeck,1
19957,unstandardized,[Memmingen,1
19958,unstandardized,Altstadt-Königsberg,1
19959,unstandardized,Polnischen Lissa,1


In [67]:
d = p(
    f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='unstandardized').rename({'place_of_publication': 'place_of_publication_unstandardized'})
    .join(
        f('all_places_of_publication').filter(c('source').is_in(['vd16','vd17','vd18']), c('place_type')=='standardized'),
        left_on=['source', 'record_number'],
        right_on=['source', 'record_number'],
        how='full'
    )
    .select(nw.coalesce(c('source'), c('source_right')), nw.coalesce(c('record_number'), c('record_number_right')), nw.coalesce(c('place_of_publication'), c('place_of_publication_unstandardized')).alias('place_of_publication'))
    .with_columns(place_of_publication=c('place_of_publication').str.replace_all('Frankfurt.*Main.*', 'Frankfurt am Main').str.replace_all('Frankfurt.*Oder.*', 'Frankfurt an der Oder'))
    .group_by(c('source'), c('place_of_publication'))
    .agg(nw.len())
    .with_columns(prop=c('len')/c('len').sum().over('source'))
    .join(f('geonames').filter(c('feature_class') == 'P').filter(c('population')==c('population').max().over('name')).select(c('name'), c('latitude'), c('longitude')), how='left', left_on='place_of_publication', right_on='name')
).to_pandas()

In [69]:
import folium
import pandas as pd
m = folium.Map(location=[48, 11], zoom_start=5)
colors = {'vd16': 'blue', 'vd17': 'green', 'vd18': 'red' }
for _, row in d.iterrows():
    if pd.notnull(row['latitude']) and pd.notnull(row['longitude']):
        folium.CircleMarker(location=[row['latitude'], row['longitude']], color=colors[row['source']] if row['source'] in colors else 'gray', radius=300*row['prop'], popup=f"{row['place_of_publication']}: {row['len']}").add_to(m)
m

In [1]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_s3 import *

  0%|          | 0/72 [00:00<?, ?it/s]

In [5]:
bnf.head(5).collect(backend='polars').to_native()

record_number,field_number,subfield_number,field_code,subfield_code,value
i32,i32,i32,str,str,str
38,2,1,"""000""","""""",""" n0 d922 45a """
493,2,1,"""000""","""""",""" n0 d922 45a """
656,2,1,"""000""","""""",""" n0 d922 45a """
861,2,1,"""000""","""""",""" n0 d922 45a """
1009,2,1,"""000""","""""",""" n0 d922 45a """


In [2]:
import hereutil
hereutil.add_to_sys_path(hereutil.here())
from src.common_basis_local import *

  0%|          | 0/74 [00:00<?, ?it/s]

In [3]:
bnf.head(5).collect(backend='polars').to_native()

record_number,field_number,subfield_number,field_code,subfield_code,value
i32,i32,i32,str,str,str
38,2,1,"""000""","""""",""" n0 d922 45a """
493,2,1,"""000""","""""",""" n0 d922 45a """
656,2,1,"""000""","""""",""" n0 d922 45a """
861,2,1,"""000""","""""",""" n0 d922 45a """
1009,2,1,"""000""","""""",""" n0 d922 45a """
